# Lección 13 - Memoria del Agente con Grafos de Conocimiento Cognee


## Configuración

Este cuaderno demuestra cómo construir un **asistente de programación** inteligente con memoria persistente usando grafos de conocimiento de [**Cognee**](https://www.cognee.ai/) y el **Microsoft Agent Framework** (MAF).

Cognee transforma texto no estructurado en un grafo de conocimiento estructurado y consultable respaldado por incrustaciones vectoriales — dando a tu agente una memoria a largo plazo rica y consciente de las relaciones.

### Lo que aprenderás
1. **Construir grafos de conocimiento**: Transformar perfiles de desarrolladores y mejores prácticas en conocimiento estructurado y consultable.
2. **Integrar Cognee con MAF**: Usar funciones `@tool` para permitir que un agente MAF consulte el grafo de conocimiento de Cognee.
3. **Conversaciones conscientes de la sesión**: Mantener contexto a través de múltiples preguntas en la misma sesión.
4. **Memoria a largo plazo**: Persistir conocimiento importante a través de sesiones y recuperarlo en nuevas conversaciones.

### Requisitos previos
- Python 3.9+
- Redis ejecutándose localmente (`docker run -d -p 6379:6379 redis`) para administración de sesiones
- Una clave API de LLM (p. ej. OpenAI) — configurar `LLM_API_KEY` en `.env`
- `CACHING=true` en `.env` (requerido para sesiones de Cognee)
- Un proyecto de Azure AI Foundry con un modelo de chat desplegado
- `AZURE_AI_PROJECT_ENDPOINT` y `AZURE_AI_MODEL_DEPLOYMENT_NAME` en `.env`
- CLI de Azure autenticado (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Tipos de Memoria del Agente

Este cuaderno explora los mismos tres tipos de memoria del cuaderno principal de la Lección 13, pero utiliza Cognee como el backend de memoria a largo plazo:

| Tipo de Memoria | Mecanismo | Duración |
|---|---|---|
| **De trabajo** | `agent.create_session()` (MAF) | Conversación única |
| **A corto plazo** | Caché de sesión de Cognee (Redis) | Sesión única |
| **A largo plazo** | Grafo de conocimiento de Cognee + vectores | Permanente |

### Arquitectura de Memoria de Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Preparar Almacenamiento Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Parte 1 — Construyendo la Base de Conocimientos

Incorporamos tres tipos de datos para crear una base de conocimientos integral para nuestro asistente de codificación:

1. **Perfil del Desarrollador** — experiencia personal y formación técnica
2. **Mejores Prácticas de Python** — el Zen de Python con directrices prácticas
3. **Conversaciones Históricas** — sesiones pasadas de preguntas y respuestas entre desarrolladores y asistentes de IA


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualiza el Grafo de Conocimiento

Cognee puede mostrar una visualización HTML interactiva de las entidades y relaciones que ha extraído.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Enriquecer la memoria con Memify

`memify()` analiza el grafo de conocimiento y genera reglas inteligentes, identificando patrones, mejores prácticas y relaciones entre conceptos.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Parte 2 — Agente MAF con Herramientas Cognee

Ahora creamos un agente MAF que puede consultar el grafo de conocimiento de Cognee mediante funciones `@tool`. Esto permite que el agente aproveche todo el poder de la búsqueda semántica consciente del grafo mientras mantiene el contexto conversacional a través de sesiones.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Memoria de trabajo con sesiones

El `AgentSession` (creado mediante `agent.create_session()`) proporciona memoria de trabajo dentro de una sesión. El agente puede referirse a mensajes anteriores mientras también consulta el grafo de conocimiento a largo plazo de Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nueva sesión — La memoria a largo plazo persiste

Iniciar una sesión nueva borra la memoria de trabajo, pero el grafo de conocimiento de Cognee sigue estando disponible. El agente puede recuperar el mismo conocimiento a largo plazo en una conversación completamente nueva.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Resumen

En este cuaderno construiste un asistente de codificación que combina la **memoria de trabajo de MAF** (`agent.create_session()`) con el **grafo de conocimiento a largo plazo de Cognee**.

### Lo que has aprendido
1. **Construcción de grafos de conocimiento**: Cognee ingiere texto no estructurado y construye un grafo + memoria vectorial.
2. **Enriquecimiento del grafo con memify**: Hechos derivados y relaciones más ricas sobre tu grafo existente.
3. **Integración MAF + Cognee**: Las funciones `@tool` permiten que un agente MAF consulte el grafo de Cognee de manera natural.
4. **Memoria de trabajo + memoria a largo plazo**: `AgentSession` (a través de `agent.create_session()`) proporciona contexto de sesión mientras Cognee ofrece conocimiento persistente.
5. **Búsqueda filtrada con NodeSets**: Apunta a subconjuntos específicos del grafo de conocimiento (p. ej. solo principios).

### Conclusiones clave
- **Cognee** convierte texto sin procesar en memoria estructurada, consciente de relaciones — más potente que una tienda vectorial simple.
- Las **funciones `@tool`** conectan limpiamente a los agentes MAF con sistemas externos de conocimiento.
- **`AgentSession`** (a través de `agent.create_session()`) mantiene el contexto por conversación separado del conocimiento a largo plazo.
- El mismo grafo de conocimiento sirve para múltiples sesiones y agentes.

### Aplicaciones en el mundo real
- **Copilotos para desarrolladores**: Revisión de código, análisis de incidentes, asistentes de arquitectura
- **Copilotos para atención al cliente**: Agentes de soporte sobre documentación de productos, FAQ y notas CRM
- **Copilotos expertos internos**: Asistentes de políticas, legales o de seguridad que razonan sobre directrices
- **Capas de datos unificadas**: Combina datos estructurados y no estructurados en un solo grafo consultable

### Próximos pasos
- Experimentar con conciencia temporal en Cognee
- Definir una ontología OWL para calidad de grafos específica del dominio
- Añadir bucles de retroalimentación del usuario para mejorar la recuperación con el tiempo
- Escalar a sistemas multiagente que compartan la misma capa de memoria de Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos hacemos responsables de malentendidos o interpretaciones erróneas derivadas del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
